# P7 Sudoku AI Lab — Google Colab

**الخطوات:** ارفعي ملف ZIP للمشروع، شغّلي الخلايا بالترتيب، درّبي CNN على MNIST الحقيقي، ثم افتحي Streamlit من الرابط الذي سيظهر. يفضّل اختيار GPU من Runtime → Change runtime type → T4 GPU.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os

# ارفعي P7_MNIST_Colab_Studio.zip من جهازك
uploaded = files.upload()
zip_name = next((name for name in uploaded if name.lower().endswith('.zip')), None)
if zip_name is None:
    raise ValueError('Please upload the project ZIP file.')
with zipfile.ZipFile(zip_name, 'r') as zf:
    zf.extractall('/content')

# Locate the folder containing train_mnist.py
PROJECT_DIR = None
for root, dirs, files_list in os.walk('/content'):
    if 'train_mnist.py' in files_list and 'app.py' in files_list:
        PROJECT_DIR = Path(root)
        break
if PROJECT_DIR is None:
    raise FileNotFoundError('Could not find the extracted project folder.')
os.chdir(PROJECT_DIR)
print('Project folder:', PROJECT_DIR)

## 1) Install the required libraries
قد يستغرق تثبيت TensorFlow وStreamlit بعض الوقت.

In [ ]:
%pip -q install -r requirements.txt

## 2) Train the CNN on real MNIST data
هذه الخلية تنزّل MNIST من Keras وتدرّب النموذج وتحفظ الدقة ومصفوفة الالتباس. لا تغلقي الجلسة أثناء التدريب.

In [ ]:
!python train_mnist.py

## 3) Start Streamlit and create a public link
لإنشاء الرابط نستخدم ngrok. ستحتاجين إلى Auth Token من حسابك في https://ngrok.com/ . أدخليه في الخانة الآمنة التالية؛ لا تشاركيه مع أحد.

In [ ]:
from getpass import getpass
from pyngrok import ngrok
import subprocess, time, requests

NGROK_TOKEN = getpass('Paste your ngrok auth token: ')
ngrok.set_auth_token(NGROK_TOKEN)

# Start Streamlit in the background
log_file = open('/content/streamlit.log', 'w')
process = subprocess.Popen(
    ['streamlit', 'run', 'app.py', '--server.port', '8501', '--server.address', '0.0.0.0', '--server.headless', 'true'],
    cwd=str(PROJECT_DIR), stdout=log_file, stderr=subprocess.STDOUT
)
time.sleep(8)
public_url = ngrok.connect(8501, 'http')
print('Open your Streamlit app here:', public_url)

## ملاحظات
- في الواجهة ستجدين دقة CNN المحسوبة من مجموعة اختبار MNIST الحقيقية بعد انتهاء التدريب.
- ارفعي صورة **رقم واحد فقط** لتجربة المصنف؛ النسخة الحالية لا تستخرج شبكة سودوكو كاملة من صورة واحدة تلقائياً.
- وضع Generated example يضيف أخطاء معروفة لاختبار Search + CSP، واحتمالاته توضيحية وليست تنبؤات CNN.
- إذا توقف الرابط، أعيدي تشغيل خلية Streamlit/ngrok.
- لإيقاف التطبيق: `ngrok.kill()` أو أوقفي جلسة Colab.